# 🧠 UCAR — Document Intelligence Pipeline (High Accuracy)
Upload any **PDF, Excel, Word, or image** file.
Uses a **3-layer accuracy system**:
- **Layer 1** — Strict extraction prompt (no approximation)
- **Layer 2** — Verification pass (cross-checks every value against source)
- **Layer 3** — Regex safety net (flags any value not found verbatim in document)

> No Tesseract. No traditional OCR. Pure LLM vision.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────
!pip install -q pymupdf pandas openpyxl python-docx pillow mistralai

In [ ]:
import os, json, base64, re, logging
from pathlib import Path

import pandas as pd
import fitz
from docx import Document
from mistralai import Mistral
from google.colab import files

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

# ── Paste your Mistral API key here ───────────────────────────────────
MISTRAL_API_KEY = "GlWXRumvGZ2BiFLlKmMpwi3lDWgUS5Qk"
client = Mistral(api_key=MISTRAL_API_KEY)

print('✅ Setup complete')

In [ ]:
# ── STEP 1 — File type detection ──────────────────────────────────────
def detect_file_type(file_path):
    ext = Path(file_path).suffix.lower()
    if ext == '.pdf':                      return 'pdf'
    elif ext in ['.xlsx', '.xls']:         return 'excel'
    elif ext == '.docx':                   return 'word'
    elif ext in ['.png', '.jpg', '.jpeg']: return 'image'
    elif ext == '.txt':                    return 'text'
    else:                                  return 'unsupported'

In [ ]:
# ── STEP 2 — File readers ─────────────────────────────────────────────

def read_pdf(file_path):
    doc = fitz.open(file_path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if text:
            pages.append({'page': i+1, 'method': 'text_layer', 'content': text})
        else:
            logger.info(f'Page {i+1} — no text layer, using vision')
            pix = page.get_pixmap(dpi=200)
            b64 = base64.b64encode(pix.tobytes('png')).decode('utf-8')
            pages.append({'page': i+1, 'method': 'vision', 'content': b64, 'mime': 'image/png'})
    return pages


def read_excel(file_path):
    xls = pd.ExcelFile(file_path)
    text, sheets = '', {}
    for sheet in xls.sheet_names:
        df = xls.parse(sheet).fillna('')
        sheets[sheet] = df.to_dict(orient='records')
        text += f'\n=== Sheet: {sheet} ===\n{df.to_string()}\n'
    return text.strip(), sheets


def read_word(file_path):
    doc = Document(file_path)
    return '\n'.join(p.text for p in doc.paragraphs if p.text.strip())


def read_image(file_path):
    ext = Path(file_path).suffix.lower().lstrip('.')
    mime = 'image/jpeg' if ext in ['jpg','jpeg'] else 'image/png'
    with open(file_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('utf-8')
    return b64, mime


def read_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

In [ ]:
# ── STEP 3 — Layer 1: Strict extraction prompt ────────────────────────

KPI_SYSTEM_PROMPT = """You are a precise data extraction AI for UCAR (University of Carthage),
a network of 30+ higher education institutions in Tunisia.
Documents may be in Arabic, French, or English — read all languages carefully.

CRITICAL ACCURACY RULES — follow these exactly:
1. Copy ALL numeric values EXACTLY as they appear — never round, estimate, or approximate
2. If the document says 9.4%, return "9.4%" — not any other number
3. If the document says 4,800,000 TND, return "4,800,000 TND" — not 4,500,000 TND
4. Extract EVERY metric you find — do not stop early, return all KPIs even if 40+
5. If you are not certain about a value, set it to null — NEVER guess or invent numbers
6. Do not invent KPIs not explicitly stated in the document
7. Only mark trend 'up' or 'down' if the document explicitly compares to a previous period

KPI domains to extract:
- academic: enrollment, graduation rates, dropout rates, pass rates, attendance, students by dept
- finance: budgets, spending, cost per student, execution rates, revenue
- hr: staff count, absenteeism, faculty ratios, vacancies, permanent vs visiting
- research: publications, patents, grants, funded projects, PhD defenses
- esg: energy, water, CO2 emissions, sustainability scores, recycling
- infrastructure: occupancy rates, equipment status, room counts, surface area
- partnerships: MOU agreements, student mobility in/out, enterprise partners
- employment: employability rates, job placement, average salary, time to employment

Return ONLY a valid JSON object — no markdown, no explanation, no code fences.

{
  "institution": "exact name from document or null",
  "period": "exact year or period from document or null",
  "document_type": "financial_report | academic_report | hr_report | research_report | general | unknown",
  "language": "Arabic | French | English | Mixed",
  "summary": "2-3 sentence plain-language summary",
  "kpis": [
    {
      "label": "KPI name in English",
      "value": "exact value with unit as written in document",
      "domain": "academic|finance|hr|research|esg|infrastructure|partnerships|employment",
      "trend": "up|down|neutral|unknown"
    }
  ],
  "insights": ["actionable insight derived from the data"],
  "alerts": ["concerning metric that needs attention"]
}"""


def llm_text(text):
    resp = client.chat.complete(
        model='mistral-large-latest',
        messages=[
            {'role': 'system', 'content': KPI_SYSTEM_PROMPT},
            {'role': 'user',   'content': f'Extract all KPIs. Copy every number exactly as written:\n\n{text[:15000]}'}
        ]
    )
    return resp.choices[0].message.content


def llm_image(b64, mime):
    resp = client.chat.complete(
        model='pixtral-large-latest',
        messages=[
            {'role': 'system', 'content': KPI_SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': [
                    {'type': 'image_url', 'image_url': {'url': f'data:{mime};base64,{b64}'}},
                    {'type': 'text', 'text': 'Read ALL text (Arabic, French, English). Copy every number exactly. Extract every KPI.'}
                ]
            }
        ]
    )
    return resp.choices[0].message.content


def parse_response(raw):
    try:
        clean = raw.strip().replace('```json','').replace('```','').strip()
        match = re.search(r'\{[\s\S]*\}', clean)
        return json.loads(match.group(0) if match else clean)
    except Exception as e:
        logger.warning(f'JSON parse failed: {e}')
        return {'summary': raw, 'kpis': [], 'insights': [], 'alerts': []}

In [ ]:
# ── STEP 4 — Layer 2: Verification pass ───────────────────────────────

VERIFY_PROMPT = """You are a data verification AI. Check extracted KPI data against the original document.

RULES:
1. Compare every numeric value in the extracted JSON against the original document
2. If a value is wrong (even slightly), correct it to match the document exactly
3. If a KPI exists in the document but is missing, add it
4. If a KPI in the extracted data does not exist in the document, remove it
5. Never invent or approximate — only use values explicitly in the document
6. Return the corrected JSON in the exact same format — no markdown, no explanation"""


def verify_extraction(original_text, extracted):
    """Second LLM pass — cross-checks every value against the source."""
    logger.info('Layer 2 — Running verification pass...')
    resp = client.chat.complete(
        model='mistral-large-latest',
        messages=[
            {'role': 'system', 'content': VERIFY_PROMPT},
            {
                'role': 'user',
                'content': (
                    f'ORIGINAL DOCUMENT:\n{original_text[:15000]}\n\n'
                    f'EXTRACTED DATA:\n{json.dumps(extracted, ensure_ascii=False)}\n\n'
                    f'Return the corrected JSON:'
                )
            }
        ]
    )
    corrected = parse_response(resp.choices[0].message.content)
    if not corrected.get('kpis'):
        logger.warning('Verification returned empty — keeping original extraction')
        return extracted
    logger.info(f'Layer 2 — Verified. KPI count: {len(corrected.get("kpis", []))}')
    return corrected

In [ ]:
# ── STEP 5 — Layer 3: Regex safety net ────────────────────────────────

def validate_numbers(original_text, extracted):
    """
    Checks every numeric value in extracted KPIs against the source.
    Flags anything that doesn't appear verbatim for human review.
    """
    flagged = []
    # Normalize source for comparison
    src = original_text.replace(' ', '').replace(',', '').replace('\n', '')

    for kpi in extracted.get('kpis', []):
        raw_value = kpi.get('value', '')
        numbers = re.findall(r'\d+[\.,]?\d*', raw_value)
        for num in numbers:
            num_clean = num.replace(',', '').replace('.', '')
            if len(num_clean) > 1 and num_clean not in src:
                flagged.append({
                    'kpi':            kpi['label'],
                    'returned_value': raw_value,
                    'suspicious_num': num,
                    'warning':        f'"{num}" not found verbatim in source — verify manually'
                })
    logger.info(f'Layer 3 — {len(flagged)} value(s) flagged')
    return flagged

In [ ]:
# ── STEP 6 — Main pipeline ────────────────────────────────────────────

def process_document(file_path):
    file_type = detect_file_type(file_path)
    file_name = os.path.basename(file_path)

    if file_type == 'unsupported':
        raise ValueError(f'Unsupported file type: {Path(file_path).suffix}')

    logger.info(f'Starting 3-layer pipeline: {file_name} [{file_type}]')
    extracted = {}
    source_text = ''

    # Images — straight to vision (no source text, skip layers 2 & 3)
    if file_type == 'image':
        b64, mime = read_image(file_path)
        logger.info('Layer 1 — Sending to Pixtral Vision...')
        extracted = parse_response(llm_image(b64, mime))
        return {'file_name': file_name, 'file_type': file_type,
                'extraction': extracted, 'flagged_values': []}

    # PDFs — hybrid per-page
    elif file_type == 'pdf':
        pages = read_pdf(file_path)
        all_kpis, all_insights, all_alerts, summaries = [], [], [], []
        institution = period = doc_type = language = None
        for page in pages:
            logger.info(f'Layer 1 — Page {page["page"]} [{page["method"]}]')
            if page['method'] == 'text_layer':
                source_text += page['content'] + '\n'
                raw = llm_text(page['content'])
            else:
                raw = llm_image(page['content'], page['mime'])
            p = parse_response(raw)
            all_kpis.extend(p.get('kpis', []))
            all_insights.extend(p.get('insights', []))
            all_alerts.extend(p.get('alerts', []))
            if p.get('summary'):  summaries.append(p['summary'])
            if not institution:   institution = p.get('institution')
            if not period:        period = p.get('period')
            if not doc_type:      doc_type = p.get('document_type')
            if not language:      language = p.get('language')
        extracted = {
            'institution':   institution, 'period': period,
            'document_type': doc_type,    'language': language,
            'summary':       ' '.join(summaries),
            'kpis':          all_kpis,
            'insights':      list(set(all_insights)),
            'alerts':        list(set(all_alerts))
        }

    # Excel
    elif file_type == 'excel':
        source_text, sheets = read_excel(file_path)
        logger.info('Layer 1 — Sending Excel to Mistral...')
        extracted = parse_response(llm_text(source_text))
        extracted['raw_sheets'] = sheets

    # Word / Text
    elif file_type in ['word', 'text']:
        source_text = read_word(file_path) if file_type == 'word' else read_text(file_path)
        logger.info('Layer 1 — Sending document to Mistral...')
        extracted = parse_response(llm_text(source_text))

    # Layer 2 — verification
    if source_text:
        extracted = verify_extraction(source_text, extracted)

    # Layer 3 — regex check
    flagged = validate_numbers(source_text, extracted) if source_text else []

    return {
        'file_name':      file_name,
        'file_type':      file_type,
        'extraction':     extracted,
        'flagged_values': flagged
    }


def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [ ]:
# ── STEP 7 — Upload & Run ─────────────────────────────────────────────
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
print(f'Uploaded: {file_path}')

In [ ]:
result = process_document(file_path)

output_file = Path(file_path).stem + '_intelligence.json'
save_json(result, output_file)

ext = result['extraction']
print('\n' + '═'*55)
print('✅  DOCUMENT PROCESSED (3-LAYER ACCURACY)')
print('═'*55)
print(f"📄  Type        : {ext.get('document_type', 'unknown')}")
print(f"🏫  Institution : {ext.get('institution', 'not detected')}")
print(f"📅  Period      : {ext.get('period', 'not detected')}")
print(f"🌍  Language    : {ext.get('language', 'not detected')}")
print(f"📊  KPIs found  : {len(ext.get('kpis', []))}")
print(f"💡  Insights    : {len(ext.get('insights', []))}")
print(f"⚠️   Alerts      : {len(ext.get('alerts', []))}")
print(f"🔍  Flagged     : {len(result.get('flagged_values', []))} value(s) need manual review")
print('─'*55)
print('\n📝 Summary:', ext.get('summary', ''))
print('\n📊 KPIs:')
for k in ext.get('kpis', []):
    trend = {'up':'↑','down':'↓','neutral':'→'}.get(k.get('trend',''), '?')
    print(f"   {trend}  [{k.get('domain','?').upper():15s}] {k.get('label','')}: {k.get('value','')}")

if ext.get('alerts'):
    print('\n⚠️  Alerts:')
    for a in ext['alerts']: print(f'   • {a}')

if ext.get('insights'):
    print('\n💡 Insights:')
    for i in ext['insights']: print(f'   • {i}')

if result.get('flagged_values'):
    print('\n🔍 Flagged values (verify manually):')
    for f in result['flagged_values']:
        print(f"   ⚠ {f['kpi']}: {f['returned_value']} — {f['warning']}")

In [ ]:
# ── Download result JSON ──────────────────────────────────────────────
files.download(output_file)
print(f'Downloaded: {output_file}')